# LungLocAI — build the training table from LUNA16 and train XGBoost

Runs the **same pipeline as the website** (3-D volume → lung extraction → nodule candidates → feature table) on every LUNA16 CT, labels the rows from `annotations.csv`, then trains the XGBoost models the website loads.

1. Upload `lunglocai-web.zip` to your Google Drive.
2. Put LUNA16 (`subset0`…`subset9` + `annotations.csv`) in Drive.
3. Run the cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- edit these paths ---
PROJECT_ZIP = '/content/drive/MyDrive/lunglocai-web.zip'
LUNA_DIR    = '/content/drive/MyDrive/LUNA16'          # contains subset0..subset9
ANNOTATIONS = f'{LUNA_DIR}/annotations.csv'
OUT_TABLE   = '/content/drive/MyDrive/lunglocai_table.csv'  # on Drive so it survives disconnects (script resumes)

In [ ]:
!unzip -qo $PROJECT_ZIP -d /content && cd /content/lunglocai-web && pip -q install -r requirements.txt

## 1. Build the LungLocAI feature table
About 10–20 s per scan. The script is **resumable** — if Colab disconnects, just run the cell again. Use `--max-scans 20` first to check everything works.

In [ ]:
%cd /content/lunglocai-web
!python training/build_luna16_table.py --luna-dir $LUNA_DIR --annotations $ANNOTATIONS --out $OUT_TABLE --workers 2

In [ ]:
import pandas as pd
t = pd.read_csv(OUT_TABLE)
print(t.shape, t.seriesuid.nunique(), 'scans')
print(t.is_nodule.value_counts())
t.head()

## 2. Train XGBoost — nodule vs. false positive
Labels come straight from LUNA16 annotations.

In [ ]:
!python training/train_xgboost.py --table $OUT_TABLE --task nodule

## 3. (Optional) Train XGBoost — lung cancer vs. benign
LUNA16 alone has **no malignancy labels**. Get them from LIDC-IDRI radiologist ratings with `training/make_lidc_malignancy_csv.py` (needs `pylidc`), rebuild the table with `--malignancy`, then train:

In [ ]:
# !python training/make_lidc_malignancy_csv.py --luna-dir $LUNA_DIR --out /content/drive/MyDrive/lidc_malignancy.csv
# !rm $OUT_TABLE   # rebuild so rows get the 'malignant' column
# !python training/build_luna16_table.py --luna-dir $LUNA_DIR --annotations $ANNOTATIONS --out $OUT_TABLE --malignancy /content/drive/MyDrive/lidc_malignancy.csv --workers 2
# !python training/train_xgboost.py --table $OUT_TABLE --task malignancy

## 4. Copy the trained models into your website
Download these files and put them in `lunglocai-web/models/` (replacing the demo models), then restart the site.

In [ ]:
!ls -la models/
from google.colab import files
for f in ['nodule_model.json','nodule_model.meta.json','malignancy_model.json','malignancy_model.meta.json']:
    import os
    if os.path.exists('models/'+f): files.download('models/'+f)